<a href="https://colab.research.google.com/github/dalia1991/from-cross-product-logic-to-simplified-QR-algorithm/blob/main/from_crossproduct_logic_to_QR_algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!sudo apt-get update
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg
!pip install manim

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libcairo2-dev is already the newest version (1.16.0-5ubuntu2.1).
libpango1.0-dev is already th

In [ ]:
# This installs the tiny version of LaTeX needed for Manim
!sudo apt-get install texlive texlive-latex-extra texlive-fonts-extra texlive-latex-recommended texlive-science texlive-fonts-essential tipa -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
E: Unable to locate package texlive-fonts-essential


In [ ]:
%%manim -v WARNING -pql QRHeartAnimation

from manim import *
import numpy as np

class QRHeartAnimation(Scene):
    def construct(self):
        # 1. Setup
        grid = NumberPlane(x_range=[-1, 7], y_range=[-1, 5])
        self.add(grid)

        vec_a_coords = np.array([4, 0, 0])
        vec_b_coords = np.array([2, 3, 0])

        arrow_a = Arrow(ORIGIN, vec_a_coords, color=YELLOW, buff=0)
        arrow_b = Arrow(ORIGIN, vec_b_coords, color=YELLOW, buff=0)

        label_a = Text("Vector A", color=YELLOW, font_size=20).next_to(arrow_a, DOWN)
        label_b = Text("Vector B", color=YELLOW, font_size=20).next_to(arrow_b, UP+RIGHT, buff=0.1)

        # Initial Area (Diamond)
        corners = [ORIGIN, vec_a_coords, vec_a_coords + vec_b_coords, vec_b_coords]
        area_diamond = Polygon(*corners, fill_color=YELLOW, fill_opacity=0.2, stroke_width=1)
        area_text = Text("Area = 12.0", color=YELLOW, font_size=28).to_corner(UL)

        self.play(GrowArrow(arrow_a), GrowArrow(arrow_b), FadeIn(label_a), FadeIn(label_b))
        self.play(FadeIn(area_diamond), Write(area_text))
        self.wait(1)

        # 2. SHOW THE PROJECTION (The part we want to remove)
        # The 'shadow' of B onto A is [2, 0, 0]
        proj_coords = np.array([2, 0, 0])
        proj_line = DashedLine(start=vec_b_coords, end=proj_coords, color=GRAY)
        proj_vector = Arrow(ORIGIN, proj_coords, color=GRAY, buff=0, stroke_width=2)
        proj_label = Text("Projection", color=GRAY, font_size=16).next_to(proj_vector, DOWN)

        self.play(Create(proj_line))
        self.play(GrowArrow(proj_vector), Write(proj_label))
        self.wait(1)

        # 3. THE SUBTRACTION (The heart of Orthogonalization)
        # B_perp = B - Projection
        b_perp_coords = np.array([0, 3, 0])
        arrow_b_perp = Arrow(ORIGIN, b_perp_coords, color=RED, buff=0)
        label_b_perp = Text("B_perp (Height = 3.0)", color=RED, font_size=20).next_to(arrow_b_perp, LEFT)

        # Rectangle corners
        rect_corners = [ORIGIN, vec_a_coords, vec_a_coords + b_perp_coords, b_perp_coords]
        rect_text = Text("Area = 4 x 3 = 12.0", color=RED, font_size=28).to_corner(UL)

        # ANIMATION: Removing the projection to get the perpendicular
        self.play(
            FadeOut(proj_vector), FadeOut(proj_label), FadeOut(proj_line),
            FadeOut(arrow_b), FadeOut(label_b),
            Transform(area_diamond, Polygon(*rect_corners, fill_color=RED, fill_opacity=0.3)),
            Transform(area_text, rect_text),
            GrowArrow(arrow_b_perp),
            Write(label_b_perp),
            run_time=3
        )

        # 4. Final QR Logic Text
        qr_logic = Text("Q matrix uses these perpendicular unit vectors", font_size=20).to_edge(DOWN)
        self.play(Create(RightAngle(arrow_a, arrow_b_perp, length=0.3)), Write(qr_logic))
        self.wait(3)

In [27]:
%%manim -v WARNING -pql QRStepByStep_GridWarp

from manim import *
import numpy as np

class QRStepByStep_GridWarp(ThreeDScene):
    def construct(self):
        # 1. Setup Camera
        self.set_camera_orientation(phi=45 * DEGREES, theta=45 * DEGREES)

        # 2. The Ghost Universe (Dashed Blue)
        axes = ThreeDAxes(axis_config={"stroke_opacity": 0.5})
        ghost_grid = NumberPlane(
            x_range=[-4, 4], y_range=[-4, 4],
            background_line_style={"stroke_color": BLUE, "stroke_opacity": 0.15}
        )
        for line in ghost_grid.get_family():
            if isinstance(line, Line): line.dashed_ratio = 0.5
        self.add(axes, ghost_grid)

        # 3. The New Universe (RGB Solid Grid)
        # We define the columns of Matrix A
        matrix_a = [
            [2, 0.5, 0.5], # New X (Red)
            [1, 2, 0.5],   # New Y (Green)
            [0, 0, 1.5]    # New Z (Blue)
        ]

        # Create a grid for each plane
        # XY Plane Grid
        new_grid = NumberPlane(
            x_range=[-3, 3], y_range=[-3, 3],
            background_line_style={"stroke_color": WHITE, "stroke_opacity": 0.4}
        )

        # 4. Starting Basis Vectors (White)
        v_x = Arrow3D(start=ORIGIN, end=[1, 0, 0], color=WHITE)
        v_y = Arrow3D(start=ORIGIN, end=[0, 1, 0], color=WHITE)
        v_z = Arrow3D(start=ORIGIN, end=[0, 0, 1], color=WHITE)

        self.add(v_x, v_y, v_z, new_grid)
        self.wait(1)

        # 5. THE GREAT WARP
        title = Text("the transformation  ", font_size=24)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        self.play(
            # Transform Basis to RGB Matrix A
            v_x.animate.set_color(RED).put_start_and_end_on(ORIGIN, [2, 0.5, 0.5]),
            v_y.animate.set_color(GREEN).put_start_and_end_on(ORIGIN, [1, 2, 0.5]),
            v_z.animate.set_color(BLUE).put_start_and_end_on(ORIGIN, [0, 0, 1.5]),
            # Warp the grid
            new_grid.animate.apply_matrix(matrix_a),
            run_time=4
        )

        # Highlight the non-orthogonality
        conclusion = Text("This is the 'Messy' Space A", color=YELLOW, font_size=20)
        self.add_fixed_in_frame_mobjects(conclusion.to_edge(DOWN))
        self.play(Write(conclusion))
        self.wait(3)

Manim Community v0.20.1

In [34]:
import numpy as np

# Your Matrix A from the code
matrix_a = np.array([
    [2, 1, 0],
    [0.5, 2, 0],
    [0.5, 0.5, 1.5]
])

# Compute QR
q, r = np.linalg.qr(matrix_a)

print("--- Matrix Q (Orthonormal Basis) ---")
print(q)
print("\n--- Matrix R (Upper Triangular) ---")
print(r)

--- Matrix Q (Orthonormal Basis) ---
[[-0.94280904  0.26086186 -0.20751434]
 [-0.23570226 -0.96192811 -0.13834289]
 [-0.23570226 -0.08151933  0.96840025]]

--- Matrix R (Upper Triangular) ---
[[-2.12132034 -1.53206469 -0.35355339]
 [ 0.         -1.70375403 -0.122279  ]
 [ 0.          0.          1.45260037]]


In [ ]:
%%manim -v WARNING -pql QR_Combined_Grid

from manim import *
import numpy as np

class QR_Combined_Grid(ThreeDScene):
    def construct(self):
        # 1. Setup Camera
        self.set_camera_orientation(phi=-45 * DEGREES, theta=45 * DEGREES)
        axes = ThreeDAxes()

        # --- DATA DEFINITIONS ---
        # Matrix Q Columns (Dark Blue) - The "Straight" Basis
        q_cols = [
            np.array([-0.9428, -0.2357, -0.2357]),
            np.array([ 0.2609, -0.9619, -0.0815]),
            np.array([-0.2075, -0.1383,  0.9684])
        ]

        # Matrix R Columns (Silver) - The "Recipe"
        r_cols = [
            np.array([-2.1213,  0.0,         0.0]),
            np.array([-1.5321, -1.7038,  0.0]),
            np.array([-0.3536, -0.1223,  1.4526])
        ]

        # Colors
        dark_blue = "#00008B"
        silver = "#C0C0C0"

        # 2. Create Vector Groups
        q_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=dark_blue) for col in q_cols
        ])

        r_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=silver) for col in r_cols
        ])

        # Labels
        label_q = Text("Q (Orthonormal Basis)", color=dark_blue, font_size=24)
        label_r = Text("R (Upper Triangular)", color=silver, font_size=24)

        # 3. Animation Sequence
        self.add(axes)

        # Show Q first
        self.add_fixed_in_frame_mobjects(label_q.to_edge(UL))
        self.play(Create(q_arrows), run_time=2)
        self.wait(1)

        # Show R on the same grid
        self.add_fixed_in_frame_mobjects(label_r.next_to(label_q, DOWN, buff=0.3))
        self.play(Create(r_arrows), run_time=2)
        self.wait(2)

        # 4. Rotate to observe alignment
        # This rotation is crucial to see that R's columns are
        # basically the original A vectors rewritten in Q's language.
        self.begin_ambient_camera_rotation(rate=1)
        self.wait(2*PI)

Manim Community v0.20.1

Waiting 4:  76%|███████▌  | 72/95 [05:16<01:42,  4.44s/it]

In [ ]:
%%manim -v WARNING -pql QR_Simplification_A2_Silver

from manim import *
import numpy as np

class QR_Simplification_A2_Silver(ThreeDScene):
    def construct(self):
        # 1. Setup Camera and Axis (Standard viewpoint)
        self.set_camera_orientation(phi=-45 * DEGREES, theta=-45 * DEGREES)
        axes = ThreeDAxes()
        self.add(axes)

        # --- Matrix A2 (Result of RQ) ---
        # The exact columns from your calculation
        a2_matrix = np.array([
            [ 2.4444,  0.9442,  0.3094],
            [ 0.4016,  1.6486,  0.1171],
            [-0.3424, -0.1184,  1.4067]
        ])

        # Colors
        silver = "#C0C0C0"

        # 2. Extract and Draw Silver Columns (Vectors)
        a2_arrows = VGroup(*[
            Arrow3D(start=ORIGIN, end=col, color=silver) for col in a2_matrix.T # .T to iterate over columns
        ])

        # Labels
        # We clarify the nature of these vectors in the title
        title = Text("Matrix A₂: Projecting Silver (R) onto Blue (Q)", color=silver, font_size=20)
        self.add_fixed_in_frame_mobjects(title.to_edge(UP))

        # 3. Animation
        self.play(Create(a2_arrows), run_time=3)
        self.wait(1)

        # 4. Final Rotation
        self.begin_ambient_camera_rotation(rate=0.2)

        # Summary text
        summary_text = Text("A₂ = RQ", font_size=32).to_edge(DOWN)
        self.add_fixed_in_frame_mobjects(summary_text)

        self.wait(5)

In [ ]:
import numpy as np

# Our evolved matrix A2
A2 = np.array([
    [ 2.4444,  0.9442,  0.3094],
    [ 0.4016,  1.6486,  0.1171],
    [-0.3424, -0.1184,  1.4067]
])

# Perform QR Factorization on A2
q2, r2 = np.linalg.qr(A2)

print("--- New Matrix Q2 (Even straighter basis) ---")
print(np.round(q2, 4))

print("\n--- New Matrix R2 (New upper triangular recipe) ---")
print(np.round(r2, 4))